In [1]:
import cv2
import numpy as np
from jetbot import Camera, Robot
from http.server import BaseHTTPRequestHandler, HTTPServer
import threading
import urllib.parse

In [2]:
def crop_height_center(frame, crop_h=300):
    height, width, _ = frame.shape

    crop_h = min(crop_h, height)

    y1 = height // 2 - crop_h // 2
    y2 = y1 + crop_h

    return frame[y1:y2, :]

In [3]:
# Function to convert BGR8 images to JPEG
def bgr8_to_jpeg(value):
    return cv2.imencode('.jpg', value)[1].tobytes()
# Initialize the camera and robot
camera = Camera.instance(width=1000, height=1000)
robot = Robot()

In [4]:
# HTTP request handler class
class CameraHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path.startswith('/camera'):
            frame = camera.value

            # Crop to 200 px high before sending
            frame = crop_height_center(frame, crop_h=200)

            self.send_response(200)
            self.send_header('Content-type', 'image/jpeg')
            self.end_headers()
            self.wfile.write(bgr8_to_jpeg(frame))
        else:
            query = urllib.parse.urlparse(self.path).query
            params = urllib.parse.parse_qs(query)

            if self.path.startswith('/set_motors'):
                left_speed = float(params.get('left', [0])[0])
                right_speed = float(params.get('right', [0])[0])
                robot.set_motors(left_speed, right_speed)
                self.send_response(200)
                self.end_headers()
                self.wfile.write(b'Motors set')

            elif self.path.startswith('/left'):
                speed = float(params.get('speed', [0])[0])
                robot.left(speed)
                self.send_response(200)
                self.end_headers()
                self.wfile.write(b'Left command executed')

            elif self.path.startswith('/right'):
                speed = float(params.get('speed', [0])[0])
                robot.right(speed)
                self.send_response(200)
                self.end_headers()
                self.wfile.write(b'Right command executed')

            elif self.path.startswith('/forward'):
                speed = float(params.get('speed', [0])[0])
                robot.forward(speed)
                self.send_response(200)
                self.end_headers()
                self.wfile.write(b'Forward command executed')

            elif self.path.startswith('/stop'):
                robot.stop()
                self.send_response(200)
                self.end_headers()
                self.wfile.write(b'Stop command executed')

In [5]:
# Function to run the HTTP server
def run_server():
    server = HTTPServer(('0.0.0.0', 8080), CameraHandler)
    server.serve_forever()
# Start the server in a separate thread
thread = threading.Thread(target=run_server)
thread.start()


194.47.156.67 - - [12/May/2026 13:53:21] "GET /camera HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:21] "GET /set_motors?left=0.12&right=0.12 HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /camera HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /set_motors?left=0.12&right=0.12 HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /camera HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /set_motors?left=0.12&right=0.12 HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /camera HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /set_motors?left=0.12&right=0.12 HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /camera HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /set_motors?left=0.12&right=0.12 HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:22] "GET /camera HTTP/1.1" 200 -
194.47.156.67 - - [12/May/2026 13:53:23] "GET /set_motors?left=0.12&right=0.12 HTTP/1.1" 200 -
194.47.156